# Inspect Matrix - Trực quan hoá Ma Trận Tương Tác
Notebook này giúp bạn xem trực tiếp ma trận train và test, đồng thời hỗ trợ lọc (filter) theo từng User hoặc từng Item.

In [1]:
import numpy as np
import pandas as pd
import os
import sys
from IPython.display import display
sys.path.append("../")
from src.data_loader import load_matrix, load_movie_titles

def load_data():
    base_dir = os.path.abspath('..')
    train_path = os.path.join(base_dir, 'data', 'processed', 'train_matrix.npy')
    test_path = os.path.join(base_dir, 'data', 'processed', 'test_matrix.npy')
    item_path = os.path.join(base_dir, 'data', 'raw', 'u.item')
    
    train = load_matrix(train_path)
    test = load_matrix(test_path)
    movie_titles = load_movie_titles(item_path)
    
    return train, test, movie_titles

train_matrix, test_matrix, movie_titles = load_data()
if train_matrix is not None:
    print(f"Train Matrix shape: {train_matrix.shape}")
    print(f"Test Matrix shape: {test_matrix.shape}")
    print(f"Total Movies with titles: {len(movie_titles)}")

Train Matrix shape: (943, 1682)
Test Matrix shape: (943, 1682)
Total Movies with titles: 1682


### 1. Lọc theo User (Xem lịch sử đánh giá của User)
Chỉ cần thay đổi tham số `user_id` và chạy lại ô bên dưới để xem kết quả của User khác.

In [2]:
def show_user_history(user_id):
    u_idx = user_id - 1
    if u_idx < 0 or u_idx >= train_matrix.shape[0]:
        print("User ID không hợp lệ!")
        return
        
    train_items = np.where(train_matrix[u_idx] > 0)[0]
    test_items = np.where(test_matrix[u_idx] > 0)[0]
    
    data = []
    for idx in train_items:
        m_id = idx + 1
        data.append({'User ID': user_id, 'Movie ID': m_id, 'Movie Title': movie_titles.get(m_id, f"Unknown ({m_id})"), 'Rating': train_matrix[u_idx, idx], 'Set': 'Train'})
    for idx in test_items:
        m_id = idx + 1
        data.append({'User ID': user_id, 'Movie ID': m_id, 'Movie Title': movie_titles.get(m_id, f"Unknown ({m_id})"), 'Rating': test_matrix[u_idx, idx], 'Set': 'Test'})
        
    df = pd.DataFrame(data)
    if not df.empty:
        df = df.sort_values(by=['Set', 'Rating'], ascending=[True, False]).reset_index(drop=True)
    
    print(f"--- LỊCH SỬ ĐÁNH GIÁ CỦA USER {user_id} ---")
    print(f"Số phim trong Train: {len(train_items)}")
    print(f"Số phim trong Test:  {len(test_items)}\n")
    if not df.empty:
        display(df)
    else:
        print("User này chưa có đánh giá nào.")

# --- Đổi user_id tại đây ---
show_user_history(user_id=1)

--- LỊCH SỬ ĐÁNH GIÁ CỦA USER 1 ---
Số phim trong Train: 227
Số phim trong Test:  45



,User ID,Movie ID,Movie Title,Rating,Set
0,1,9,Dead Man Walking (1995),5.0,Test
1,1,89,Blade Runner (1982),5.0,Test
2,1,96,Terminator 2: Judgment Day (1991),5.0,Test
3,1,115,"Haunted World of Edward D. Wood Jr., The (1995)",5.0,Test
4,1,129,Bound (1996),5.0,Test
...,...,...,...,...,...
267,1,247,Turbo: A Power Rangers Movie (1997),1.0,Train
268,1,259,George of the Jungle (1997),1.0,Train
269,1,260,Event Horizon (1997),1.0,Train
270,1,261,Air Bud (1997),1.0,Train


### 2. Lọc theo Item (Xem danh sách User đã đánh giá phim)
Chỉ cần thay đổi tham số `item_id` và chạy lại ô bên dưới để xem kết quả của Phim khác.

In [3]:
def show_item_history(item_id):
    i_idx = item_id - 1
    if i_idx < 0 or i_idx >= train_matrix.shape[1]:
        print("Movie ID không hợp lệ!")
        return
        
    train_users = np.where(train_matrix[:, i_idx] > 0)[0]
    test_users = np.where(test_matrix[:, i_idx] > 0)[0]
    
    data = []
    for u_idx in train_users:
        u_id = u_idx + 1
        data.append({'Movie ID': item_id, 'Movie Title': movie_titles.get(item_id, f"Unknown ({item_id})"), 'User ID': u_id, 'Rating': train_matrix[u_idx, i_idx], 'Set': 'Train'})
    for u_idx in test_users:
        u_id = u_idx + 1
        data.append({'Movie ID': item_id, 'Movie Title': movie_titles.get(item_id, f"Unknown ({item_id})"), 'User ID': u_id, 'Rating': test_matrix[u_idx, i_idx], 'Set': 'Test'})
        
    df = pd.DataFrame(data)
    if not df.empty:
        df = df.sort_values(by=['Set', 'Rating'], ascending=[True, False]).reset_index(drop=True)
    
    m_title = movie_titles.get(item_id, f"Unknown ({item_id})")
    print(f"--- LỊCH SỬ ĐÁNH GIÁ CỦA PHIM: {m_title} (ID: {item_id}) ---")
    print(f"Số lượt xem trong Train: {len(train_users)}")
    print(f"Số lượt xem trong Test:  {len(test_users)}\n")
    if not df.empty:
        display(df)
    else:
        print("Phim này chưa có đánh giá nào.")

# --- Đổi item_id tại đây ---
show_item_history(item_id=1)

--- LỊCH SỬ ĐÁNH GIÁ CỦA PHIM: Toy Story (1995) (ID: 1) ---
Số lượt xem trong Train: 365
Số lượt xem trong Test:  87



,Movie ID,Movie Title,User ID,Rating,Set
0,1,Toy Story (1995),38,5.0,Test
1,1,Toy Story (1995),43,5.0,Test
2,1,Toy Story (1995),93,5.0,Test
3,1,Toy Story (1995),130,5.0,Test
4,1,Toy Story (1995),151,5.0,Test
...,...,...,...,...,...
447,1,Toy Story (1995),424,1.0,Train
448,1,Toy Story (1995),463,1.0,Train
449,1,Toy Story (1995),609,1.0,Train
450,1,Toy Story (1995),761,1.0,Train
